# Inspect VLS — single-shot spectra, binned

Single-shot Gotthard spectra → optional filtering → average spectra in bins of any per-shot quantity
(GMD, VLS integral, centre of mass, width, OPIS, time), with optional normalisation.
Sections 5 and 6 check the moment extraction: binning on the **centre of mass** should shift the
averaged spectra along with the bin, binning on the **width** should broaden them.

`COMBINED_H5 = None` reads the raw FLASH files of `RUN`; otherwise a combined file from `write_h5.py`.
The per-shot data end up in plain arrays: `S` (single-shot spectra, `(n_shots, n_pixels)`), `px_roi`
(pixel axis) and the dict `shots` of per-shot values — edit and add to them freely.

In [ ]:
# ---- Parameters ---------------------------------------------------------------
RUN = 60513
COMBINED_H5 = None           # None = raw FLASH files of RUN; else combined file (relative to config.COMBINED_DIR or absolute)
TRAIN_STEP = 20              # every N-th train (VLS ~0.5 MB per train)
MAX_FILES = None             # raw mode: first N raw files only
VLS_OFFSET = None            # raw mode: Gotthard line b+offset -> bunch b; None = config.VLS_OFFSET
BUNCH_RANGE = None           # (start, stop) bunch slots used; None = slots where the GMD shows pulses

VLS_ROI = (480, 640)         # pixels holding the spectrum (half-open) — moves with photon energy
VLS_BG_PIXELS = (650, 800)   # signal-free pixels: per-shot common-mode offset
VLS_BAD_PIXELS = [0, 1, 679, 680, 681, *range(832, 864)]
MOMENT_THRESHOLD = None      # ADU: pixels below are ignored (set to 0) for COM / width; None = use all

# Filters on per-shot values: (lo, hi) or None. Keys of the `shots` dict (section 2).
FILTERS = {"gmd": None, "vls_sum": None, "com": None, "width": None, "t_min": None}

# Binning of the averaged spectra (section 4)
BIN_ON = "gmd"               # key of `shots`: "gmd", "vls_sum", "com", "width", "mpe", "t_min", ...
N_BINS = 8
BIN_PERCENTILES = (2, 98)    # edges spread evenly between these percentiles of BIN_ON ...
BIN_EDGES = None             # ... unless explicit edges are given, e.g. np.linspace(4, 16, 7)
NORMALISE = None             # None: mean spectrum | "gmd": sum(spectra)/sum(GMD) per bin | "area": unit area

In [ ]:
# ---- Imports and paths ------------------------------------------------------
from pathlib import Path
import sys
import warnings

import h5py
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

# Repo root = the first folder above the working directory that holds analysis/scripts/config.py
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "analysis" / "scripts" / "config.py").exists())
sys.path.insert(0, str(REPO_ROOT / "analysis" / "scripts"))
import binning
import config
import data_loading
import processing

# Empty bunch slots are NaN by design, so these numpy warnings are expected noise
for msg in ("All-NaN slice", "Mean of empty slice", "invalid value encountered", "divide by zero",
            "Degrees of freedom"):
    warnings.filterwarnings("ignore", message=msg)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})
TRAIN_RATE_HZ = 10

## 1. Load

In [ ]:
if COMBINED_H5 is None:
    data = data_loading.load_raw_h5(RUN, config=1, train_step=TRAIN_STEP, max_files=MAX_FILES,
                                    vls_offset=VLS_OFFSET)
    source = f"run {RUN} (every {TRAIN_STEP}. train)"
else:
    h5_path = Path(COMBINED_H5)
    if not h5_path.is_absolute():
        h5_path = config.COMBINED_DIR / h5_path
    data = data_loading.load_data(str(h5_path), config=1, downsample_N=TRAIN_STEP)
    source = f"{h5_path.name} (every {TRAIN_STEP}. train)"
assert data.vls is not None, "no VLS in this data"

gmd = data.gmd               # (n_trains, n_bunches) µJ, NaN = no pulse
mpe = data.mpe               # (n_trains,) eV
vls = data.vls               # (n_trains, n_bunches, 1280) ADU, NaN = no Gotthard line
vls[..., VLS_BAD_PIXELS] = np.nan
t_min = (data.tID - data.tID[0]) / TRAIN_RATE_HZ / 60
n_trains, n_bunches = gmd.shape
print(source, f"| {n_trains} trains x {n_bunches} bunch slots")

## 2. Single-shot spectra and moments

Pedestal = mean frame of the slots that have a Gotthard line but no FEL pulse; each shot also has its
common-mode offset (median over `VLS_BG_PIXELS`) removed, then the ROI is kept. Moments come from
`processing.spectral_moments`: integral, centre of mass and rms width (pixels).

In [ ]:
pulse = np.isfinite(gmd)
has_frame = np.isfinite(vls[:, :, VLS_BG_PIXELS[0]])
if BUNCH_RANGE is None:
    lit_slots = np.flatnonzero(pulse.mean(axis=0) > 0.5)
    BUNCH_RANGE = (int(lit_slots[0]), int(lit_slots[-1]) + 1)
shot = pulse & has_frame
shot[:, :BUNCH_RANGE[0]] = False
shot[:, BUNCH_RANGE[1]:] = False

# pedestal from dark frames (line but no pulse)
dark = has_frame & ~pulse
if dark.sum() > 100:
    pedestal = np.zeros(vls.shape[-1])
    for b in np.flatnonzero(dark.any(axis=0)):
        pedestal += vls[dark[:, b], b].sum(axis=0)
    pedestal /= dark.sum()
else:
    pedestal = np.zeros(vls.shape[-1])
    print("WARNING: no dark frames — no pedestal subtraction, common mode only")

px_roi = np.arange(*VLS_ROI)
spec = vls[:, :, slice(*VLS_ROI)] - pedestal[slice(*VLS_ROI)]
spec -= np.nanmedian(vls[:, :, slice(*VLS_BG_PIXELS)] - pedestal[slice(*VLS_BG_PIXELS)], axis=-1)[..., None]
S = np.nan_to_num(spec[shot])                      # (n_shots, n_pixels) single-shot spectra
n_shots = len(S)

S_mom = S if MOMENT_THRESHOLD is None else np.where(S > MOMENT_THRESHOLD, S, 0)
_, com, width = processing.spectral_moments(S_mom, px_roi)

train_idx, slot_idx = np.nonzero(shot)             # where each shot came from
shots = {
    "gmd": gmd[shot],
    "vls_sum": S.sum(axis=1),
    "com": com,
    "width": width,
    "mpe": mpe[train_idx],
    "t_min": t_min[train_idx],
    "slot": slot_idx,
}
print(f"{n_shots} shots, bunch slots {BUNCH_RANGE}, ROI {VLS_ROI} ({len(px_roi)} px)")

In [ ]:
# Distributions of the per-shot values; shaded = filter range
fig, axs = plt.subplots(1, 4, figsize=(17, 3.5))
for ax, name, unit in zip(axs, ["gmd", "vls_sum", "com", "width"], ["µJ", "ADU", "pixel", "pixel"]):
    v = shots[name][np.isfinite(shots[name])]
    ax.hist(v, bins=np.linspace(*np.percentile(v, [0.5, 99.5]), 100), histtype="step")
    if FILTERS.get(name) is not None:
        ax.axvspan(*FILTERS[name], color="C1", alpha=0.15)
    ax.set(xlabel=f"{name} ({unit})", ylabel="shots")
fig.suptitle(source)
fig.tight_layout()
plt.show()

In [ ]:
# A few strong single shots with their extracted centre of mass (dashed) and ± width (shaded)
strongest = np.argsort(shots["vls_sum"])[::-1]
examples = np.random.default_rng(0).choice(strongest[:max(n_shots // 10, 8)], 8, replace=False)
fig, axs = plt.subplots(2, 4, figsize=(17, 6), sharex=True)
for ax, i in zip(axs.flat, examples):
    ax.plot(px_roi, S[i], lw=0.8)
    ax.axvline(com[i], color="C3", ls="--")
    ax.axvspan(com[i] - width[i], com[i] + width[i], color="C3", alpha=0.12)
    ax.set_title(f"shot {i}: GMD {shots['gmd'][i]:.1f} µJ", fontsize=9)
for ax in axs[1]:
    ax.set_xlabel("pixel")
for ax in axs[:, 0]:
    ax.set_ylabel("ADU")
fig.tight_layout()
plt.show()

## 3. Filter

In [ ]:
keep = np.ones(n_shots, bool)
for name, rng in FILTERS.items():
    if rng is not None:
        lo, hi = rng
        keep &= (shots[name] >= lo) & (shots[name] <= hi)
        print(f"{name} in [{lo}, {hi}]: {keep.sum()} shots left")
keep &= np.isfinite(shots[BIN_ON])            # shots without a value can't be binned
print(f"kept {keep.sum()} of {n_shots} shots")

## 4. Average spectra binned on `BIN_ON`

In [ ]:
def edges_for(x, n_bins=None, percentiles=None):
    # BIN_EDGES if given, else n_bins bins spread evenly between two percentiles of x
    if BIN_EDGES is not None and n_bins is None:
        return np.asarray(BIN_EDGES, dtype=float)
    lo, hi = percentiles or BIN_PERCENTILES
    return np.nanpercentile(x, np.linspace(lo, hi, (n_bins or N_BINS) + 1))


def colours(n, cmap="viridis"):
    return mpl.colormaps[cmap](np.linspace(0, 0.95, n))


def bin_spectra(x, edges, spectra, g, normalise=NORMALISE):
    # per-bin average spectrum (rows) — mean, sum/sum(GMD) or unit-area — and shots per bin
    cents, in_bin = binning.arb_bool_ar(edges, x)
    if normalise == "gmd":
        avg, n = binning.bin_and_sum_ratio(in_bin, spectra, g)
    else:
        avg, _, n = binning.bin_and_average(in_bin, spectra)
        if normalise == "area":
            avg = avg / avg.sum(axis=1, keepdims=True)
    return cents, avg, n

In [ ]:
x = shots[BIN_ON][keep]
edges = edges_for(x)
cents, avg, n_in = bin_spectra(x, edges, S[keep], shots["gmd"][keep])
print("shots per bin:", n_in.tolist())

ylabel = {None: "mean ADU", "gmd": "ADU / µJ", "area": "normalised"}[NORMALISE]
fig, axs = plt.subplots(1, 2, figsize=(15, 4.5))
for c, spec_bin, col in zip(cents, avg, colours(len(cents))):
    axs[0].plot(px_roi, spec_bin, color=col, lw=1)
sm = mpl.cm.ScalarMappable(mpl.colors.Normalize(cents.min(), cents.max()), "viridis")
fig.colorbar(sm, ax=axs[0], label=BIN_ON)
axs[0].set(xlabel="pixel", ylabel=ylabel, title=f"binned on {BIN_ON}, {source}")
px_edges = np.append(px_roi, px_roi[-1] + 1) - 0.5           # pixel bin edges for pcolormesh
mesh = axs[1].pcolormesh(px_edges, edges, avg, shading="flat", cmap="magma")
fig.colorbar(mesh, ax=axs[1], label=ylabel)
axs[1].set(xlabel="pixel", ylabel=BIN_ON)
fig.tight_layout()
plt.show()

## 5. Moment check — centre of mass

Bins on the single-shot COM. If the extraction works, each bin's averaged spectrum sits at its bin
(dashed lines) and the COM of the averaged spectrum follows the bin centre (right, on the diagonal).
A COM pulled towards the ROI centre means noise / baseline in the ROI dominates — try
`MOMENT_THRESHOLD` or a narrower `VLS_ROI`.

In [ ]:
N_CHECK_BINS = 8
x = shots["com"][keep]
edges_c = edges_for(x, N_CHECK_BINS, (5, 95))
cents_c, avg_c, n_c = bin_spectra(x, edges_c, S[keep], shots["gmd"][keep], normalise="area")
_, com_of_avg, _ = processing.spectral_moments(avg_c, px_roi)

fig, axs = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [1.6, 1]})
for c, spec_bin, col in zip(cents_c, avg_c, colours(len(cents_c), "plasma")):
    axs[0].plot(px_roi, spec_bin, color=col, lw=1.2, label=f"{c:.0f}")
    axs[0].axvline(c, color=col, ls="--", lw=0.8)
axs[0].set(xlabel="pixel", ylabel="area-normalised", title="average spectrum per single-shot COM bin")
axs[0].legend(title="COM bin", fontsize=8, ncol=2)
axs[1].plot(cents_c, com_of_avg, "o-", label="COM of the averaged spectrum")
axs[1].plot(cents_c, cents_c, "k--", lw=0.8, label="bin centre (y = x)")
axs[1].set(xlabel="single-shot COM bin (pixel)", ylabel="pixel")
axs[1].legend()
fig.tight_layout()
plt.show()

## 6. Moment check — width

Bins on the single-shot width. Each shot is first shifted so its COM sits at the ROI centre (otherwise
the COM jitter would broaden every bin alike); averaged spectra should then get broader with the bin,
and their width should follow the bin centre. Pure noise over the ROI gives a width of about
`ROI / √12` (dotted line).

In [ ]:
# shift every shot (integer pixels) so its COM lands on the ROI centre
centre = px_roi.mean()
shift = np.nan_to_num(np.round(com - centre)).astype(int)
idx = np.clip(np.arange(len(px_roi))[None, :] + shift[:, None], 0, len(px_roi) - 1)
S_aligned = np.take_along_axis(S, idx, axis=1)

x = shots["width"][keep]
edges_w = edges_for(x, N_CHECK_BINS, (5, 95))
cents_w, avg_w, n_w = bin_spectra(x, edges_w, S_aligned[keep], shots["gmd"][keep], normalise="area")
_, _, width_of_avg = processing.spectral_moments(avg_w, px_roi)

fig, axs = plt.subplots(1, 2, figsize=(15, 4.5), gridspec_kw={"width_ratios": [1.6, 1]})
for c, spec_bin, col in zip(cents_w, avg_w, colours(len(cents_w), "plasma")):
    axs[0].plot(px_roi - centre, spec_bin, color=col, lw=1.2, label=f"{c:.0f}")
axs[0].set(xlabel="pixel − COM", ylabel="area-normalised", title="COM-aligned average spectrum per width bin")
axs[0].legend(title="width bin", fontsize=8, ncol=2)
axs[1].plot(cents_w, width_of_avg, "o-", label="width of the aligned average")
axs[1].plot(cents_w, cents_w, "k--", lw=0.8, label="bin centre (y = x)")
axs[1].axhline(len(px_roi) / np.sqrt(12), color="gray", ls=":", label="noise over ROI")
axs[1].set(xlabel="single-shot width bin (pixel)", ylabel="pixel")
axs[1].legend()
fig.tight_layout()
plt.show()